# YOLO training - thesis dataset step2/step3

Use this notebook in Google Colab with GPU enabled.

Expected files in Google Drive:

`/content/drive/MyDrive/thesis/dataset_step3_tiny_person_crops.zip`

Recommended previous weights, based on the first benchmark:

`/content/drive/MyDrive/thesis/models/YOLO26s_best.pt`

`/content/drive/MyDrive/thesis/models/YOLO26n_best.pt`

The zip must contain the selected dataset folder with `data.yaml`, `images/`, and `labels/`.

In [ ]:
!nvidia-smi
!python --version

In [ ]:
!pip install -q ultralytics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import shutil
import yaml

DATASET_NAME = 'dataset_step3_tiny_person_crops'
DATA_ZIP = Path(f'/content/drive/MyDrive/thesis/{DATASET_NAME}.zip')
DATA_ROOT = Path('/content/datasets')
DATA_DIR = DATA_ROOT / DATASET_NAME
RUNS_DIR = Path('/content/drive/MyDrive/thesis/runs')

assert DATA_ZIP.exists(), f'Missing dataset zip: {DATA_ZIP}'

if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

!unzip -q "{DATA_ZIP}" -d "{DATA_ROOT}"

assert (DATA_DIR / 'data.yaml').exists(), f'Missing data.yaml in {DATA_DIR}'
assert (DATA_DIR / 'images/train').exists(), 'Missing train images'
assert (DATA_DIR / 'labels/train').exists(), 'Missing train labels'

print('Dataset ready:', DATA_DIR)

In [ ]:
DATA_YAML = DATA_DIR / 'data_colab.yaml'

with open(DATA_DIR / 'data.yaml', 'r') as f:
    data = yaml.safe_load(f)

data['path'] = str(DATA_DIR)
data['train'] = 'images/train'
data['val'] = 'images/val'
data['test'] = 'images/test'

with open(DATA_YAML, 'w') as f:
    yaml.safe_dump(data, f, sort_keys=False)

print(DATA_YAML.read_text())

In [ ]:
from collections import Counter

names = data['names']

def count_split(split):
    counts = Counter()
    labels = sorted((DATA_DIR / 'labels' / split).glob('*.txt'))
    images = list((DATA_DIR / 'images' / split).glob('*'))
    for label in labels:
        for line in label.read_text().splitlines():
            parts = line.split()
            if parts:
                counts[names[int(float(parts[0]))]] += 1
    return len(images), len(labels), counts

for split in ['train', 'val', 'test']:
    image_count, label_count, counts = count_split(split)
    print(split, 'images=', image_count, 'labels=', label_count, dict(counts))

In [ ]:
from ultralytics import YOLO

# Your first benchmark shows YOLO26s is the best accuracy candidate,
# and YOLO26n is the best speed/size candidate.
# Put the previous best weights in Google Drive under MyDrive/thesis/models/.

EXPERIMENTS = {
    'main_accuracy': {
        'model': '/content/drive/MyDrive/thesis/models/YOLO26s_best.pt',
        'epochs': 80,
        'imgsz': 640,
        'run_name': 'step2_citypersons_yolo26s_finetune',
    },
    'person_recall_960': {
        'model': '/content/drive/MyDrive/thesis/models/YOLO26s_best.pt',
        'epochs': 40,
        'imgsz': 960,
        'run_name': 'step2_citypersons_yolo26s_person_recall_960',
        'patience': 10,
        'optimizer': 'AdamW',
        'lr0': 0.0002,
        'lrf': 0.1,
        'mosaic': 0.2,
        'close_mosaic': 5,
    },
    'person_recall_800': {
        'model': '/content/drive/MyDrive/thesis/models/YOLO26s_best.pt',
        'epochs': 40,
        'imgsz': 800,
        'run_name': 'step2_citypersons_yolo26s_person_recall_800',
        'patience': 10,
        'optimizer': 'AdamW',
        'lr0': 0.0002,
        'lrf': 0.1,
        'mosaic': 0.2,
        'close_mosaic': 5,
    },
    'realtime_light': {
        'model': '/content/drive/MyDrive/thesis/models/YOLO26n_best.pt',
        'epochs': 60,
        'imgsz': 640,
        'run_name': 'step2_citypersons_yolo26n_realtime_640',
        'patience': 10,
        'optimizer': 'AdamW',
        'lr0': 0.0002,
        'lrf': 0.1,
        'mosaic': 0.2,
        'close_mosaic': 5,
    },
    'realtime_light_800': {
        'model': '/content/drive/MyDrive/thesis/models/YOLO26n_best.pt',
        'epochs': 50,
        'imgsz': 800,
        'run_name': 'step2_citypersons_yolo26n_realtime_800',
        'patience': 10,
        'optimizer': 'AdamW',
        'lr0': 0.0002,
        'lrf': 0.1,
        'mosaic': 0.2,
        'close_mosaic': 5,
    },
    'realtime_light_800_step3': {
        'model': '/content/drive/MyDrive/thesis/models/YOLO26n_best.pt',
        'epochs': 60,
        'imgsz': 800,
        'run_name': 'step3_tiny_person_yolo26n_realtime_800',
        'patience': 12,
        'optimizer': 'AdamW',
        'lr0': 0.0002,
        'lrf': 0.1,
        'mosaic': 0.2,
        'close_mosaic': 5,
    },
    'standard_baseline': {
        'model': 'yolov8s.pt',
        'epochs': 80,
        'imgsz': 640,
        'run_name': 'step2_citypersons_yolov8s_baseline',
    },
}

SELECTED_EXPERIMENT = 'realtime_light_800_step3'
cfg = EXPERIMENTS[SELECTED_EXPERIMENT]

MODEL = cfg['model']
EPOCHS = cfg['epochs']
IMGSZ = cfg['imgsz']
RUN_NAME = cfg['run_name']
PATIENCE = cfg.get('patience', 20)

if MODEL.endswith('.pt') and MODEL.startswith('/content/'):
    assert Path(MODEL).exists(), f'Missing model weights: {MODEL}'

print('Selected experiment:', SELECTED_EXPERIMENT)
print('Model:', MODEL)
print('Run:', RUN_NAME)

model = YOLO(MODEL)
train_kwargs = dict(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=-1,
    patience=PATIENCE,
    workers=2,
    device=0,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
)

for key in ['optimizer', 'lr0', 'lrf', 'mosaic', 'close_mosaic']:
    if key in cfg:
        train_kwargs[key] = cfg[key]

print('Train overrides:', {k: train_kwargs[k] for k in train_kwargs if k in cfg})
results = model.train(**train_kwargs)

In [ ]:
best_pt = RUNS_DIR / RUN_NAME / 'weights' / 'best.pt'
assert best_pt.exists(), best_pt

best_model = YOLO(str(best_pt))
val_metrics = best_model.val(data=str(DATA_YAML), split='val', imgsz=IMGSZ, device=0, plots=True)
test_metrics = best_model.val(data=str(DATA_YAML), split='test', imgsz=IMGSZ, device=0, plots=True)

print('Best model:', best_pt)

In [ ]:
onnx_path = best_model.export(format='onnx', imgsz=IMGSZ, opset=12, simplify=True)
print('Exported ONNX:', onnx_path)

After training, download or keep these files in Google Drive:

- `runs/step2_citypersons_yolo26s_finetune/weights/best.pt`
- `runs/step2_citypersons_yolo26s_finetune/weights/best.onnx`
- `runs/step2_citypersons_yolo26s_finetune/results.csv`
- confusion matrix and PR curves from the run folder

Then compare class metrics for `person`, `truck`, `emergency_vehicle`, and `bus` before adding more datasets.